# Case Study 3: E-Commerce Customer Segmentation
**Methods:** RFM, K-Means, DBSCAN, PCA, t-SNE

## Objective
Build customer segments from online-retail transactions, compare K-Means and DBSCAN, visualize groups using PCA and t-SNE, and suggest marketing personas based on measured cluster profiles.

Dataset: UCI Online Retail. The notebook attempts to download the public dataset if it is not already in `data/Online Retail.xlsx`.


In [1]:
# If needed, run in a notebook cell:
# %pip install -r ../requirements.txt
import os, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

SEED = 42
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FILE = DATA_DIR / "Online Retail.xlsx"
if not FILE.exists():
    try:
        archive = DATA_DIR / "online_retail.zip"
        urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/352/online+retail.zip", archive)
        with zipfile.ZipFile(archive) as z:
            xlsx = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
            with z.open(xlsx) as src, open(FILE, "wb") as dst:
                dst.write(src.read())
    except Exception as e:
        raise RuntimeError("Could not download dataset. Download UCI Online Retail manually and save as data/Online Retail.xlsx") from e

raw = pd.read_excel(FILE)
display(raw.head())
print("Original shape:", raw.shape)


ModuleNotFoundError: No module named 'pandas'

## 1. Clean transactions and create RFM features
- Recency: days since last purchase; smaller is more recent.
- Frequency: distinct invoices/orders.
- Monetary: total spend, calculated as Quantity × UnitPrice.
Cancelled invoices, missing customer IDs, and non-positive quantity/price rows are removed.


In [ ]:
required = {"InvoiceNo","InvoiceDate","Quantity","UnitPrice","CustomerID"}
assert required.issubset(raw.columns), f"Missing columns: {required - set(raw.columns)}"
df = raw.dropna(subset=["CustomerID","InvoiceNo","InvoiceDate"]).copy()
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df["CustomerID"] = df["CustomerID"].astype(str).str.replace(r"\.0$", "", regex=True)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["MonetaryLine"] = df["Quantity"] * df["UnitPrice"]
snapshot = df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda s: (snapshot - s.max().normalize()).days),
    Frequency=("InvoiceNo","nunique"),
    Monetary=("MonetaryLine","sum")
).reset_index()
rfm = rfm[(rfm["Frequency"] > 0) & (rfm["Monetary"] > 0)].copy()
display(rfm.describe().round(2))
print("Customers:", len(rfm))


## 2. Explore and scale RFM
RFM distributions are skewed, so use log1p and standard scaling for clustering. Original RFM values remain available for interpretation.


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, col in zip(axes, ["Recency","Frequency","Monetary"]):
    sns.histplot(rfm[col], bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

X_log = np.log1p(rfm[["Recency","Frequency","Monetary"]])
scaler = StandardScaler()
X = scaler.fit_transform(X_log)


## 3. K-Means and selection of K
Silhouette score is used to compare K values from 2 to 6. A high score suggests better separation, but also inspect cluster sizes and business usefulness.


In [ ]:
scores = {}
for k in range(2,7):
    labels = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(X)
    scores[k] = silhouette_score(X, labels)
display(pd.DataFrame({"K":list(scores.keys()),"Silhouette":list(scores.values())}))
best_k = max(scores, key=scores.get)
rfm["KMeans_Cluster"] = KMeans(n_clusters=best_k, random_state=SEED, n_init=10).fit_predict(X)
print("Best K by silhouette:", best_k)
pd.Series(rfm["KMeans_Cluster"]).value_counts().sort_index().plot(kind="bar", title="K-Means cluster sizes")
plt.xlabel("Cluster"); plt.ylabel("Customers"); plt.show()


## 4. DBSCAN
DBSCAN groups dense points and labels noise as `-1`. The parameters below are initial settings and should be reviewed based on resulting cluster counts.


In [ ]:
db = DBSCAN(eps=0.7, min_samples=10)
rfm["DBSCAN_Cluster"] = db.fit_predict(X)
display(rfm["DBSCAN_Cluster"].value_counts().sort_index().rename("Customers").to_frame())
labels = rfm["DBSCAN_Cluster"].to_numpy()
mask = labels != -1
if len(set(labels[mask])) >= 2:
    print("DBSCAN silhouette excluding noise:", round(silhouette_score(X[mask], labels[mask]), 4))
else:
    print("DBSCAN found fewer than two non-noise clusters; silhouette is not applicable.")


## 5. Compare cluster profiles and suggest personas
Profiles use original, untransformed RFM values. Persona labels are heuristics—not true labels—and should be checked against the displayed statistics.


In [ ]:
def profile(col):
    return rfm.groupby(col).agg(
        Customers=("CustomerID","nunique"),
        Recency_Median=("Recency","median"),
        Frequency_Median=("Frequency","median"),
        Monetary_Median=("Monetary","median"),
        Monetary_Mean=("Monetary","mean")
    ).round(2)

print("K-Means profiles"); display(profile("KMeans_Cluster"))
print("DBSCAN profiles (cluster -1 is noise)"); display(profile("DBSCAN_Cluster"))

med = rfm[["Recency","Frequency","Monetary"]].median()
def persona(row):
    if row["Recency_Median"] <= med["Recency"] and row["Frequency_Median"] >= med["Frequency"] and row["Monetary_Median"] >= med["Monetary"]:
        return "Loyal/high-value", "Reward loyalty, VIP access, and early product previews."
    if row["Recency_Median"] > med["Recency"] and row["Frequency_Median"] < med["Frequency"]:
        return "At-risk/inactive", "Test a win-back message or time-limited reactivation offer."
    if row["Frequency_Median"] <= med["Frequency"] and row["Monetary_Median"] >= med["Monetary"]:
        return "High-spend/occasional", "Recommend complementary products and encourage repeat purchases."
    if row["Recency_Median"] <= med["Recency"] and row["Frequency_Median"] < med["Frequency"]:
        return "Recent/developing", "Use onboarding and a second-purchase incentive."
    return "Regular/mid-value", "Use relevant recommendations and monitor engagement."

persona_rows = []
for cid, row in profile("KMeans_Cluster").iterrows():
    name, action = persona(row)
    persona_rows.append({"Cluster":cid, "Persona":name, "Suggested_Action":action, **row.to_dict()})
display(pd.DataFrame(persona_rows))


## 6. PCA visualization
PCA projects the scaled RFM features to two principal components. Clustering itself used the scaled three-feature data.


In [ ]:
pca = PCA(n_components=2, random_state=SEED)
Xp = pca.fit_transform(X)
rfm["PCA1"], rfm["PCA2"] = Xp[:,0], Xp[:,1]
print("PCA explained variance ratio:", pca.explained_variance_ratio_.round(4))
print("Total explained:", round(pca.explained_variance_ratio_.sum(),4))
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.scatterplot(data=rfm,x="PCA1",y="PCA2",hue="KMeans_Cluster",palette="tab10",s=20,linewidth=0,ax=axes[0])
axes[0].set_title("K-Means clusters — PCA")
sns.scatterplot(data=rfm,x="PCA1",y="PCA2",hue="DBSCAN_Cluster",palette="tab10",s=20,linewidth=0,ax=axes[1])
axes[1].set_title("DBSCAN clusters — PCA")
plt.tight_layout(); plt.show()


## 7. t-SNE visualization
t-SNE is nonlinear and preserves local neighborhood structure. Do not interpret large distances between groups as precise global distances. A sample is used to keep runtime reasonable.


In [ ]:
n = min(3000, len(rfm))
sample_idx = rfm.sample(n=n, random_state=SEED).index
Xs = X[rfm.index.get_indexer(sample_idx)]
perplexity = min(30, max(2, (n-1)//3))
Xt = TSNE(n_components=2, random_state=SEED, perplexity=perplexity, init="pca", learning_rate="auto").fit_transform(Xs)
plot_df = rfm.loc[sample_idx, ["KMeans_Cluster","DBSCAN_Cluster"]].copy()
plot_df["tSNE1"], plot_df["tSNE2"] = Xt[:,0], Xt[:,1]
sns.scatterplot(data=plot_df,x="tSNE1",y="tSNE2",hue="KMeans_Cluster",palette="tab10",s=20,linewidth=0)
plt.title("K-Means clusters — t-SNE sample"); plt.tight_layout(); plt.show()


## 8. Conclusion prompts
Write conclusions using the outputs generated by your own run:
1. Which K gave the highest silhouette score?
2. How many non-noise clusters did DBSCAN find, and what percentage of customers were noise?
3. What RFM patterns distinguish each cluster?
4. Which marketing action fits each persona?
5. Limitations: RFM ignores demographics/product preferences; results depend on cleaning, time window, scaling, and hyperparameters. Test marketing actions before rollout.


In [ ]:
noise_fraction = (rfm["DBSCAN_Cluster"] == -1).mean()
print(f"Selected K: {best_k}")
print(f"DBSCAN non-noise clusters: {len(set(rfm['DBSCAN_Cluster']) - {-1})}")
print(f"DBSCAN noise fraction: {noise_fraction:.1%}")
